In [ ]:
import torch
import matplotlib.pyplot as plt
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders
from model import Decoder_Only_Transformer

PATH = "Obras_completas_ilustradas_Federico_Garcia_Lorca_djvu.txt"
PERMITTED = set("abcdefghijklmnñopqrstuvwxyzáéíóúü"
               "ABCDEFGHIJKLMNÑOPQRSTUVWXYZÁÉÍÓÚÜ"
               " \n.,;:¡!¿?-'\"()")

def load_text(path):
    with open(path, encoding="utf-8") as f:
        return "".join(c for c in f.read() if c in PERMITTED)


def BPETokenizer_train(text, vocab_size):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False) #base tokens Byte level
    tok.decoder = decoders.ByteLevel()
    tok.train_from_iterator([text], trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    return tok


def CharTokenizer_train(text):
    chars = sorted(set(text))  #base tokens
    stoi = {c: i for i, c in enumerate(chars)}
    return torch.tensor([stoi[c] for c in text]), len(stoi)


def tokenize(text, bpe_vocab=None):
    if bpe_vocab is None:
        return CharTokenizer_train(text)
    tok = BPETokenizer_train(text, bpe_vocab)
    return torch.tensor(tok.encode(text).ids), tok.get_vocab_size()


def batches(data, n_batches, batch_size, block_size):
    for _ in range(n_batches):
        ix = torch.randint(len(data) - block_size, (batch_size,))
        x = torch.stack([data[i:i + block_size] for i in ix])
        y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
        yield x, y


@torch.no_grad()
def evaluate(model, data, batch_size, iters=20):
    model.eval()
    losses = [model(x, y)[1] for x, y in batches(data, iters, batch_size, model.block_size)]
    model.train()
    return torch.stack(losses).mean().item()


def train(model, train_data, val_data, steps, batch_size, eval_every=10):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    history = []

    for step, (x, y) in enumerate(batches(train_data, steps, batch_size, model.block_size)):
        if step % eval_every == 0:
            train_loss = evaluate(model, train_data, batch_size)
            val_loss = evaluate(model, val_data, batch_size)

            history.append((train_loss, val_loss))

            print(
                f"step {step}: "
                f"train loss {train_loss:.4f}, "
                f"val loss {val_loss:.4f}"
            )

        _, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return history


In [ ]:
text = load_text(PATH) #loading the text and cleaning it up to only include characters in our vocab
data_chars, vocab_size_chars = tokenize(text, 500)
data_bpe, vocab_size_bpe = tokenize(text, 512)
n = int(0.9 * len(data_bpe))
train_data = data_bpe[:n]
val_data = data_bpe[n:]
steps = 2000
model = Decoder_Only_Transformer(vocab_size=vocab_size_bpe, block_size=64, d_model=64, n_heads=4, n_layers=4)

In [ ]:
block_size=6
pos = torch.arange(block_size)


In [ ]:
print(pos.unsqueeze(0))

In [ ]:
train(model, train_data, val_data, steps=steps, batch_size=32, eval_every=1)

In [ ]:
model

In [ ]:
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)
x_data
x_data.dtype

In [ ]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
x_np.dtype

In [ ]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # retains the properties of x_datax
print(f"Random Tensor: \n {x_rand} \n")

In [ ]:
shape = (2, 3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

In [ ]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

In [ ]:
# We move our tensor to the GPU if available
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
tensor = tensor.to(device)
print(f"Device tensor is stored on: {tensor.device}")

In [ ]:
tensor = torch.ones(4, 4)
tensor[:,1] = 0
print(tensor)

In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

In [ ]:
# This computes the element-wise product
print(f"tensor.mul(tensor) \n {tensor.mul(tensor)} \n")
# Alternative syntax:
print(f"tensor * tensor \n {tensor * tensor}")

In [ ]:
print(f"tensor.matmul(tensor.T) \n {tensor.matmul(tensor.T)} \n")
# Alternative syntax:
print(f"tensor @ tensor.T \n {tensor @ tensor.T}")

In [ ]:
print(tensor, "\n")
tensor.add_(5)
print(tensor)

In [2]:
!curl -o quijote.txt https://babel.upm.es/~angel/teaching/pps/quijote.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2091k  100 2091k    0     0  1881k      0  0:00:01  0:00:01 --:--:-- 1882k
